# 02.04 · Trees & ensembles

**Chapter:** Machine Learning Fundamentals · **Milestone:** M1

## Goals
- [ ] Understand decision trees and why a single tree overfits
- [ ] Bagging (random forest) vs boosting (gradient boosting)
- [ ] Compare the boosting libraries: scikit-learn `HistGradientBoosting` vs XGBoost vs LightGBM (vs CatBoost)

## Resources
- [StatQuest – Decision trees, random forests, gradient boost (YouTube)](https://www.youtube.com/@statquest) — free
- [XGBoost docs](https://xgboost.readthedocs.io/) · [LightGBM docs](https://lightgbm.readthedocs.io/) · [CatBoost docs](https://catboost.ai/docs/) — free

On **tabular data**, gradient-boosted trees still usually beat neural nets and LLMs. An AI engineer who reaches for
GPT to classify spreadsheet rows when LightGBM would do it in 50 ms for free is a cautionary tale.

> macOS: LightGBM and XGBoost need OpenMP: `brew install libomp`.

In [ ]:
import time

import matplotlib.pyplot as plt
import numpy as np
from sklearn.datasets import make_classification
from sklearn.ensemble import HistGradientBoostingClassifier, RandomForestClassifier
from sklearn.linear_model import LogisticRegression
from sklearn.model_selection import train_test_split
from sklearn.tree import DecisionTreeClassifier, plot_tree

X, y = make_classification(n_samples=30_000, n_features=30, n_informative=12, n_redundant=6,
                           n_clusters_per_class=4, class_sep=0.8, flip_y=0.03, random_state=0)
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.25, random_state=0)

## 1. A decision tree
Recursively split on the feature/threshold that best separates classes (lowest Gini impurity / entropy).
Deep trees memorise the training set.

In [ ]:
for depth in [2, 5, 10, None]:
    t = DecisionTreeClassifier(max_depth=depth, random_state=0).fit(X_train, y_train)
    print(f"max_depth={depth}: train {t.score(X_train, y_train):.3f} | test {t.score(X_test, y_test):.3f}")

plt.figure(figsize=(12, 4))
plot_tree(DecisionTreeClassifier(max_depth=2).fit(X_train, y_train), filled=True, fontsize=8); plt.show()

## 2. Ensembles
- **Bagging / random forest**: many deep trees on bootstrap samples + random feature subsets, averaged. Reduces variance.
- **Boosting**: trees added sequentially, each fixing the previous ones' errors (fitting the gradient of the loss). Reduces bias.

In [ ]:
import lightgbm as lgb
import xgboost as xgb

models = {
    "logistic regression (baseline)": LogisticRegression(max_iter=2000),
    "random forest": RandomForestClassifier(n_estimators=300, n_jobs=-1, random_state=0),
    "sklearn HistGradientBoosting": HistGradientBoostingClassifier(max_iter=300, random_state=0),
    "XGBoost": xgb.XGBClassifier(n_estimators=300, learning_rate=0.1, max_depth=6, n_jobs=-1),
    "LightGBM": lgb.LGBMClassifier(n_estimators=300, learning_rate=0.1, num_leaves=63, verbose=-1),
}
results = {}
for name, m in models.items():
    start = time.perf_counter(); m.fit(X_train, y_train); secs = time.perf_counter() - start
    results[name] = (m.score(X_test, y_test), secs)
    print(f"{name:32} acc={results[name][0]:.3f}  fit={secs:.1f}s")

| Library | Notes |
|---|---|
| scikit-learn `HistGradientBoosting*` | no extra dependency, LightGBM-style algorithm, good default |
| **XGBoost** | the classic; GPU support; huge ecosystem |
| **LightGBM** | very fast on large data; leaf-wise growth |
| **CatBoost** | best native handling of categorical features; robust defaults |

## 3. Early stopping & feature importance
Boosting overfits if you add too many trees. Hold out a validation set and stop when it stops improving.

In [ ]:
X_tr, X_val, y_tr, y_val = train_test_split(X_train, y_train, test_size=0.2, random_state=0)
booster = lgb.LGBMClassifier(n_estimators=5000, learning_rate=0.05, num_leaves=63, verbose=-1)
booster.fit(X_tr, y_tr, eval_set=[(X_val, y_val)], callbacks=[lgb.early_stopping(50, verbose=False)])
print("best number of trees:", booster.best_iteration_, "| test acc:", round(booster.score(X_test, y_test), 3))

imp = booster.feature_importances_
order = np.argsort(imp)[::-1][:15]
plt.bar([f"f{i}" for i in order], imp[order]); plt.title("LightGBM split importance (top 15)"); plt.show()

## Exercises
1. `uv add catboost` and add CatBoost to the comparison.
2. Use a real tabular dataset: `sklearn.datasets.fetch_openml("adult", version=2, as_frame=True)` (has categorical columns).
   Compare one-hot encoding + XGBoost vs LightGBM/CatBoost native categoricals.
3. Tune LightGBM's `num_leaves` and `learning_rate` with a small grid on the validation set. How much does tuning buy over defaults?

In [ ]:
# your code here